In [301]:
import pandas as pd
import numpy as np

from IPython.display import display

In [302]:
staging = pd.read_csv('staging_dataset_cmu 1.csv')
claims_1 = pd.read_csv('claims_staged_cmu_1_2.csv')
claims_2 = pd.read_csv('claims_staged_cmu_2_2.csv')

# merge two claims datasets
claims = pd.concat([claims_1, claims_2], ignore_index=True)


# I. Staging

## A. Staging Descriptive Stat

In [303]:
staging

,ICD10_CODE,member_number,MOST_RECENT_PATH_STAGE_DT,MOST_RECENT_CLINICAL_STAGE_DT,PATHOLOGIC_STAGE_GROUP,CLINICAL_STAGE_GROUP
0,C50.411,B908,64.000,NaN,Stage IA,NaN
1,C50.919,B760,NaN,888.000,NaN,Stage IA
2,C34.11,C597,NaN,0.000,NaN,Stage IVB
3,C50.519,B621,77.000,NaN,Stage IA,NaN
4,C50.412,A392,33.000,NaN,Stage IA,NaN
...,...,...,...,...,...,...
3762,C20,D338,NaN,-2168.000,NaN,Stage IIA
3763,C50.812,B004,NaN,238.000,NaN,Stage IIA
3764,C21.0,D206,NaN,11.000,NaN,Stage IIA
3765,C18.2,B320,20.000,NaN,Stage IVA,NaN


In [304]:
staging.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3767 entries, 0 to 3766
Data columns (total 6 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   ICD10_CODE                     3767 non-null   object 
 1   member_number                  3767 non-null   object 
 2   MOST_RECENT_PATH_STAGE_DT      1913 non-null   float64
 3   MOST_RECENT_CLINICAL_STAGE_DT  2313 non-null   float64
 4   PATHOLOGIC_STAGE_GROUP         1853 non-null   object 
 5   CLINICAL_STAGE_GROUP           2223 non-null   object 
dtypes: float64(2), object(4)
memory usage: 176.7+ KB


In [305]:
# missing values
staging.isna().sum().sort_values(ascending=False)

PATHOLOGIC_STAGE_GROUP           1914
MOST_RECENT_PATH_STAGE_DT        1854
CLINICAL_STAGE_GROUP             1544
MOST_RECENT_CLINICAL_STAGE_DT    1454
ICD10_CODE                          0
member_number                       0
dtype: int64

In [306]:
# percentage missingness
((staging.isna().sum() / len(staging)) * 100).sort_values(ascending=False)

PATHOLOGIC_STAGE_GROUP          50.810
MOST_RECENT_PATH_STAGE_DT       49.217
CLINICAL_STAGE_GROUP            40.988
MOST_RECENT_CLINICAL_STAGE_DT   38.598
ICD10_CODE                       0.000
member_number                    0.000
dtype: float64

In [307]:
# number of unique values per column
print(staging.nunique()) 

'''3657 records in staging'''
'''Need to check path/clinical stages for multiple records'''

ICD10_CODE                         64
member_number                    3657
MOST_RECENT_PATH_STAGE_DT         754
MOST_RECENT_CLINICAL_STAGE_DT     869
PATHOLOGIC_STAGE_GROUP             20
CLINICAL_STAGE_GROUP               21
dtype: int64


'Need to check path/clinical stages for multiple records'

## B. Check multiple staging records

In [308]:
# how many member numbers have multiple records in staging
print('Number of members with multiple records:', (staging['member_number'].value_counts() > 1).sum())

'''Among all 3,657 members, 107 of them got multiple staging records'''

# check among them, what is the distribution of records
vc = staging['member_number'].value_counts()
dup_members = vc[vc > 1].index  # member_numbers that appear more than once

# Show all columns and all rows for these members, as a full DataFrame in the notebook
display(staging[staging['member_number'].isin(dup_members)])

'''These 107 members have multiple staging records may contribute to stage and claims analysis'''

# List out these 107 members (dup_members) with their number of records
for member, count in vc[vc > 1].items():
    print(f"Member: {member} - Number of records: {count}")

'''See below for full listing of all 107 members with multiple staging records and their number of records.

Members with 3 records:
D391, C353, B102

(The rest have 2 records; full list shown below)
B320, C871, C239, A536, C555, A664, C100, C978, A233, A082, C453, D090, A380, A254, C020, D122, C859, B233, C794, C534, C440, C698, B899, C326, D471, B354, C134, A792, B091, A123, B575, C454, B248, D070, C350, D575, B201, A286, B238, C390, A065, A505, C608, A929, B641, C466, A447, B409, A956, A032, A464, D420, B696, B097, B473, D351, A124, C081, B054, D579, B311, C377, D519, A809, B926, C927, D047, A622, A643, B406, B940, A135, B373, C145, C791, C314, C186, B752, C866, A597, C090, B514, D363, B576, B172, D324, A023, B424, A759, B156, B466, C032, B845, A236, A567, C899, B402, D071, C885, A108, B154, B057, A700, A050

(Full explicit member list printed for clarity)'''

Number of members with multiple records: 107


,ICD10_CODE,member_number,MOST_RECENT_PATH_STAGE_DT,MOST_RECENT_CLINICAL_STAGE_DT,PATHOLOGIC_STAGE_GROUP,CLINICAL_STAGE_GROUP
31,C34.32,B057,NaN,1242.000,NaN,Stage IA3
32,C34.90,B057,NaN,840.000,NaN,Stage IIB
133,C50.211,A464,61.000,NaN,Stage IA,NaN
134,C50.412,A464,61.000,NaN,Stage IA,NaN
145,C34.11,A108,61.000,28.000,Stage IIIA,Stage IIIA
...,...,...,...,...,...,...
3712,C50.912,A700,NaN,992.000,NaN,Stage IA
3725,C50.911,A050,NaN,21.000,NaN,Stage IA
3726,C50.912,A050,194.000,NaN,Stage IA,NaN
3765,C18.2,B320,20.000,NaN,Stage IVA,NaN


Member: D391 - Number of records: 3
Member: C353 - Number of records: 3
Member: B102 - Number of records: 3
Member: B320 - Number of records: 2
Member: C871 - Number of records: 2
Member: C239 - Number of records: 2
Member: A536 - Number of records: 2
Member: C555 - Number of records: 2
Member: A664 - Number of records: 2
Member: C100 - Number of records: 2
Member: C978 - Number of records: 2
Member: A233 - Number of records: 2
Member: A082 - Number of records: 2
Member: C453 - Number of records: 2
Member: D090 - Number of records: 2
Member: A380 - Number of records: 2
Member: A254 - Number of records: 2
Member: C020 - Number of records: 2
Member: D122 - Number of records: 2
Member: C859 - Number of records: 2
Member: B233 - Number of records: 2
Member: C794 - Number of records: 2
Member: C534 - Number of records: 2
Member: C440 - Number of records: 2
Member: C698 - Number of records: 2
Member: B899 - Number of records: 2
Member: C326 - Number of records: 2
Member: D471 - Number of rec

'See below for full listing of all 107 members with multiple staging records and their number of records.\n\nMembers with 3 records:\nD391, C353, B102\n\n(The rest have 2 records; full list shown below)\nB320, C871, C239, A536, C555, A664, C100, C978, A233, A082, C453, D090, A380, A254, C020, D122, C859, B233, C794, C534, C440, C698, B899, C326, D471, B354, C134, A792, B091, A123, B575, C454, B248, D070, C350, D575, B201, A286, B238, C390, A065, A505, C608, A929, B641, C466, A447, B409, A956, A032, A464, D420, B696, B097, B473, D351, A124, C081, B054, D579, B311, C377, D519, A809, B926, C927, D047, A622, A643, B406, B940, A135, B373, C145, C791, C314, C186, B752, C866, A597, C090, B514, D363, B576, B172, D324, A023, B424, A759, B156, B466, C032, B845, A236, A567, C899, B402, D071, C885, A108, B154, B057, A700, A050\n\n(Full explicit member list printed for clarity)'

In [309]:
# Take a look at member D391, C353, C102
display(staging[staging['member_number'].isin(['D391', 'C353', 'B102'])])

# There are 3 members with multiple records in staging, two of them have more than one cancer type

,ICD10_CODE,member_number,MOST_RECENT_PATH_STAGE_DT,MOST_RECENT_CLINICAL_STAGE_DT,PATHOLOGIC_STAGE_GROUP,CLINICAL_STAGE_GROUP
971,C34.12,C353,NaN,-555.000,NaN,Stage IA1
972,C50.911,C353,-870.000,NaN,Stage IB,NaN
973,C50.919,C353,-945.000,NaN,Stage IB,NaN
1890,C20,D391,114.000,-46.000,Stage IIA,Stage IIIB
1891,C34.91,D391,NaN,2194.000,NaN,Stage IA2
1892,C34.92,D391,1238.000,1211.000,Stage IA3,Stage IA2
2960,C34.11,B102,43.000,NaN,Stage IA2,NaN
2961,C34.31,B102,NaN,685.000,NaN,Stage IA1
2962,C34.32,B102,NaN,1590.000,NaN,Stage IA2


In [310]:
# how many member numbers have exactly 1 record in staging
print('Number of members with exactly 1 record:', (staging['member_number'].value_counts() == 1).sum())

# how many member numbers have >1 record in staging
print('Number of members with >1 record:', (staging['member_number'].value_counts() > 1).sum())

# verify total number of unique members: (members with exactly 1 record) + (members with >1 record)
print('Total number of unique members:', (staging['member_number'].value_counts() == 1).sum() + (staging['member_number'].value_counts() > 1).sum())


Number of members with exactly 1 record: 3550
Number of members with >1 record: 107
Total number of unique members: 3657


## C. Mapping ICD10 to cancers

In [311]:
# group by first 3 characters of ICD10 code and count occurrences
staging['ICD10_group'] = staging['ICD10_CODE'].str[:3]
icd10_group_counts = staging['ICD10_group'].value_counts()
print(icd10_group_counts)
print(icd10_group_counts.sum()) # number of records in staging

'''
Clarification: 
Number of records (3767) - numbers of unique members (3657) = 110, but why is that members with multiple records is 107?
A: 107 unique members who have multiple records have 110 staging records
'''

ICD10_group
C50    2300
C34     715
C18     501
C20     171
C21      55
C19      25
Name: count, dtype: int64
3767


'\nClarification: \nNumber of records (3767) - numbers of unique members (3657) = 110, but why is that members with multiple records is 107?\nA: 107 unique members who have multiple records have 110 staging records\n'

In [312]:
# detailed mapping of ICD10 codes to cancer types
detailed_cancer_map = {
    "C50": "breast",
    "C34": "lung",
    "C18": "colon",
    "C19": "rectal",      # rectosigmoid
    "C20": "rectal",      # rectum
    "C21": "anal"         # anus (or keep as colorectal)
}

'''
treatments for colon and rectal cancers are different, so we create a new column for detailed cancer type
'''

staging['detailed_cancer_type'] = staging['ICD10_group'].map(detailed_cancer_map)

# creating a new column for cancer type, C18, C19, C20, C21 are all colorectal
broad_cancer_map = {
    "C50": "breast",
    "C34": "lung",
    "C18": "colorectal",
    "C19": "colorectal",
    "C20": "colorectal",
    "C21": "colorectal"
}
staging['cancer_type'] = staging['ICD10_group'].map(broad_cancer_map)

In [313]:
staging.head()

,ICD10_CODE,member_number,MOST_RECENT_PATH_STAGE_DT,MOST_RECENT_CLINICAL_STAGE_DT,PATHOLOGIC_STAGE_GROUP,CLINICAL_STAGE_GROUP,ICD10_group,detailed_cancer_type,cancer_type
0,C50.411,B908,64.000,NaN,Stage IA,NaN,C50,breast,breast
1,C50.919,B760,NaN,888.000,NaN,Stage IA,C50,breast,breast
2,C34.11,C597,NaN,0.000,NaN,Stage IVB,C34,lung,lung
3,C50.519,B621,77.000,NaN,Stage IA,NaN,C50,breast,breast
4,C50.412,A392,33.000,NaN,Stage IA,NaN,C50,breast,breast


## D. Explore dt and path & clinical staging

In [314]:
# summary statistics for numeric columns
staging.describe() # what does negative path stage mean?


# 0: member's first cancer-related diagnosis


,MOST_RECENT_PATH_STAGE_DT,MOST_RECENT_CLINICAL_STAGE_DT
count,1913.000,2313.000
mean,112.602,133.441
std,679.545,650.935
min,-3089.000,-3106.000
25%,23.000,0.000
50%,54.000,32.000
75%,152.000,153.000
max,3482.000,3613.000


In [315]:
# crosstab of pathologic and clinical staging
staging["has_path"] = staging["PATHOLOGIC_STAGE_GROUP"].notna()
staging["has_clin"] = staging["CLINICAL_STAGE_GROUP"].notna()

pd.crosstab(staging["has_path"], staging["has_clin"])

has_clin,False,True
has_path,,
False,0,1914
True,1544,309


In [316]:
# verify that sum of crosstab is equal to number of numbers of staging records
crosstab_counts = pd.crosstab(staging["has_path"], staging["has_clin"])
display(crosstab_counts)
total = crosstab_counts.values.sum()
print('Sum of all 4 cells:', total)

has_clin,False,True
has_path,,
False,0,1914
True,1544,309


Sum of all 4 cells: 3767


In [317]:
# show patients with both pathological and clinical staging
both = staging[
    staging["PATHOLOGIC_STAGE_GROUP"].notna() &
    staging["CLINICAL_STAGE_GROUP"].notna()
].copy()


In [318]:
# check for members with both path and clin staging and if their stages match
both["stages_match"] = (
    both["PATHOLOGIC_STAGE_GROUP"].str.strip() ==
    both["CLINICAL_STAGE_GROUP"].str.strip()
)

both["stages_match"].value_counts() # count of matches

# among those members (309) with both path and clin staging, 212 have matching stages

stages_match
False    212
True      97
Name: count, dtype: int64

In [319]:
# create a new column for final stage 
staging["FINAL_STAGE"] = (
    staging["CLINICAL_STAGE_GROUP"]
    .combine_first(staging["PATHOLOGIC_STAGE_GROUP"])
)

# Highmark prefers clinical staging over pathologic staging

In [320]:
# group by pathologic stage group
# staging.groupby('PATHOLOGIC_STAGE_GROUP').describe()['MOST_RECENT_PATH_STAGE_DT']

In [321]:
# group by clinical stage group
# staging.groupby('CLINICAL_STAGE_GROUP').describe()['MOST_RECENT_CLINICAL_STAGE_DT']

In [322]:
# number of unique members per final stage group
staging.groupby("FINAL_STAGE")["member_number"].nunique().sort_values(ascending=False)

FINAL_STAGE
Stage IA                1217
Stage IIA                414
Stage IIIB               358
Stage IB                 343
Stage IIB                269
Stage IV                 195
Stage IIIA               192
Stage IA2                130
Stage IVA                121
Stage IIIC               104
Stage I                  100
Stage IVB                 82
Stage IA3                 56
No Stage Recommended      52
Stage IA1                 34
Stage IVC                 26
Stage 0                   19
Stage Unknown             11
Stage IIC                  8
Stage II                   3
Occult carcinoma           1
Name: member_number, dtype: int64

In [323]:
# distinct final stage
print(staging['FINAL_STAGE'].unique())

print('Number of distinct final stage:', staging['FINAL_STAGE'].nunique())

['Stage IA' 'Stage IVB' 'Stage IA2' 'Stage IB' 'Stage IIA' 'Stage IIB'
 'Stage IIIB' 'Stage IV' 'No Stage Recommended' 'Stage IA3' 'Stage IVA'
 'Stage IIIA' 'Stage I' 'Stage 0' 'Stage IIIC' 'Stage IIC' 'Stage IVC'
 'Stage IA1' 'Stage Unknown' 'Occult carcinoma' 'Stage II']
Number of distinct final stage: 21


In [324]:
# simple stage mapping
Simple_stage_map = {
    "Stage 0": "Stage 0", #
    "Stage I": "Stage 1", #
    "Stage IA": "Stage 1", #
    "Stage IA1": "Stage 1", #
    "Stage IA2": "Stage 1", #
    "Stage IA3": "Stage 1", #
    "Stage IB": "Stage 1", #
    "Stage II": "Stage 2", #
    "Stage IIA": "Stage 2", #
    "Stage IIB": "Stage 2", #
    "Stage IIC": "Stage 2", #
    "Stage IIIA": "Stage 3", #
    "Stage IIIB": "Stage 3", #
    "Stage IIIC": "Stage 3", #
    "Stage IV": "Stage 4", #
    "Stage IVA": "Stage 4", #
    "Stage IVB": "Stage 4", #
    "Stage IVC": "Stage 4", #
    "No Stage Recommended": "No Stage",
    "Stage Unknown": "No Stage",
    "Occult carcinoma": "No Stage",   
    
}

# add a column for simple stage info
staging['simple_stage'] = staging['FINAL_STAGE'].map(Simple_stage_map)
staging.head()

,ICD10_CODE,member_number,MOST_RECENT_PATH_STAGE_DT,MOST_RECENT_CLINICAL_STAGE_DT,PATHOLOGIC_STAGE_GROUP,CLINICAL_STAGE_GROUP,ICD10_group,detailed_cancer_type,cancer_type,has_path,has_clin,FINAL_STAGE,simple_stage
0,C50.411,B908,64.000,NaN,Stage IA,NaN,C50,breast,breast,True,False,Stage IA,Stage 1
1,C50.919,B760,NaN,888.000,NaN,Stage IA,C50,breast,breast,False,True,Stage IA,Stage 1
2,C34.11,C597,NaN,0.000,NaN,Stage IVB,C34,lung,lung,False,True,Stage IVB,Stage 4
3,C50.519,B621,77.000,NaN,Stage IA,NaN,C50,breast,breast,True,False,Stage IA,Stage 1
4,C50.412,A392,33.000,NaN,Stage IA,NaN,C50,breast,breast,True,False,Stage IA,Stage 1


In [325]:
# number of records with no stage
print('Number of records with no stage:', len(staging[staging['simple_stage'] == 'No Stage']))

# drop records with unclear/no stage info
staging = staging[staging['simple_stage'] != 'No Stage']
print(staging.groupby("simple_stage")["member_number"].nunique().sort_values(ascending=False))

# verify number of records after dropping unclear/no stage info
print('Number of records after dropping unclear/no stage info:', len(staging))
print('Number of unique members after dropping unclear/no stage info:', len(staging['member_number'].unique()))

staging.head()

# Remove records with unclear/no stage info
# 3767 - 64 = 3703

Number of records with no stage: 64
simple_stage
Stage 1    1861
Stage 2     694
Stage 3     651
Stage 4     423
Stage 0      19
Name: member_number, dtype: int64
Number of records after dropping unclear/no stage info: 3703
Number of unique members after dropping unclear/no stage info: 3597


,ICD10_CODE,member_number,MOST_RECENT_PATH_STAGE_DT,MOST_RECENT_CLINICAL_STAGE_DT,PATHOLOGIC_STAGE_GROUP,CLINICAL_STAGE_GROUP,ICD10_group,detailed_cancer_type,cancer_type,has_path,has_clin,FINAL_STAGE,simple_stage
0,C50.411,B908,64.000,NaN,Stage IA,NaN,C50,breast,breast,True,False,Stage IA,Stage 1
1,C50.919,B760,NaN,888.000,NaN,Stage IA,C50,breast,breast,False,True,Stage IA,Stage 1
2,C34.11,C597,NaN,0.000,NaN,Stage IVB,C34,lung,lung,False,True,Stage IVB,Stage 4
3,C50.519,B621,77.000,NaN,Stage IA,NaN,C50,breast,breast,True,False,Stage IA,Stage 1
4,C50.412,A392,33.000,NaN,Stage IA,NaN,C50,breast,breast,True,False,Stage IA,Stage 1


### Decide to drop members with duplicate staging records or duplicate cancer
Current: drop all 107 member (dup_member: those have > 1 staging records) with 110 staging records 
-> may drop members with only one cancer and its stage update 

OR

Alternaitve: drop dup_member who have > 1 cancer type

In [326]:
# dup_member: those have > 1 staging records
# multi_records_members = staging["member_number"].value_counts() ... 


In [327]:
# drop dup_member who have > 1 cancer type
multi_cancer_members = staging.groupby("member_number")["cancer_type"].nunique()
multi_cancer_members = multi_cancer_members[multi_cancer_members > 1].index
print('Number of records of dup_member who have > 1 cancer type:', len(staging[staging['member_number'].isin(multi_cancer_members)]))
print('Number of unique members with multiple cancer types:', len(multi_cancer_members))

# drop dup_member who have > 1 cancer type
staging = staging[~staging["member_number"].isin(multi_cancer_members)]

# verify number of records and unique members after dropping dup_member with multiple cancer types
print('Number of records after dropping dup_member with multiple cancer types:', len(staging)) # 3703 - 62 = 3641
print('Number of unique members after dropping dup_member with multiple cancer types:', len(staging['member_number'].unique())) # 3597 - 30 = 3567


Number of records of dup_member who have > 1 cancer type: 62
Number of unique members with multiple cancer types: 30
Number of records after dropping dup_member with multiple cancer types: 3641
Number of unique members after dropping dup_member with multiple cancer types: 3567


In [328]:
# after dropping dup_member who have > 1 cancer type
# how many member numbers have exactly 1 record in staging
print('Number of members with exactly 1 record:', (staging['member_number'].value_counts() == 1).sum())

# how many member numbers have >1 record in staging
print('Number of members with >1 record:', (staging['member_number'].value_counts() > 1).sum())


Number of members with exactly 1 record: 3494
Number of members with >1 record: 73


# II. Claims

In [329]:
claims.head()

,PRI_DIAG_CD,ADM_DIAG_CD,DIAG_CD_001,DIAG_CD_002,DIAG_CD_003,HCPCS,DRG,Med_Cost_Category,NDC_CD,DRUG_NM,C_UTIL_CT,c_allowed,days_since_earliest_dt,member_number,provider_number,ROUTE DESCRIPTION,STRENGTH DESCRIPTION,PROC_DESC,DIAG_DESC,DRG_DESC,ADM_DIAG_DESC,DIAG_001_DIAG_DESC,DIAG_002_DIAG_DESC,DIAG_003_DIAG_DESC
0,J431,NaN,NaN,NaN,NaN,94727,NaN,Prof PCP Procedures,NaN,NaN,1.000,18.500,-364,A001,NaN,NaN,NaN,Pulm function test by gas,Panlobular emphysema,NaN,NaN,NaN,NaN,NaN
1,J431,NaN,NaN,NaN,NaN,94060,NaN,Prof PCP Procedures,NaN,NaN,1.000,20.000,-364,A001,NaN,NaN,NaN,Evaluation of wheezing,Panlobular emphysema,NaN,NaN,NaN,NaN,NaN
2,J431,NaN,NaN,NaN,NaN,94060,NaN,OP Other,NaN,NaN,1.000,458.120,-364,A001,2236.000,NaN,NaN,Evaluation of wheezing,Panlobular emphysema,NaN,NaN,NaN,NaN,NaN
3,J431,NaN,NaN,NaN,NaN,94726,NaN,OP Other,NaN,NaN,0.000,0.000,-364,A001,2236.000,NaN,NaN,Pulm funct tst plethysmograp,Panlobular emphysema,NaN,NaN,NaN,NaN,NaN
4,J431,NaN,NaN,NaN,NaN,94729,NaN,OP Other,NaN,NaN,0.000,0.000,-364,A001,2236.000,NaN,NaN,Co/membane diffuse capacity,Panlobular emphysema,NaN,NaN,NaN,NaN,NaN


In [330]:
# check missing values
claims.isna().sum().sort_values(ascending=False)

DRG_DESC                  2512525
DRG                       2512496
ADM_DIAG_DESC             2507683
ADM_DIAG_CD               2507535
STRENGTH DESCRIPTION      1785960
ROUTE DESCRIPTION         1785960
NDC_CD                    1753429
DRUG_NM                   1752403
DIAG_003_DIAG_DESC        1680037
DIAG_CD_003               1662254
DIAG_002_DIAG_DESC        1432755
DIAG_CD_002               1408647
DIAG_001_DIAG_DESC        1047844
DIAG_CD_001               1022916
provider_number            868329
PROC_DESC                  577304
HCPCS                      522687
DIAG_DESC                  384833
PRI_DIAG_CD                354145
member_number                   0
c_allowed                       0
C_UTIL_CT                       0
Med_Cost_Category               0
days_since_earliest_dt          0
dtype: int64

In [331]:
# percentage missingness
((claims.isna().sum() / len(claims)) * 100).sort_values(ascending=False)

# member_number is PK for claims
# provider_number: around 1/3 of the records are missing provider number
# Inpatient related column: DRG
# 

DRG_DESC                 99.807
DRG                      99.806
ADM_DIAG_DESC            99.615
ADM_DIAG_CD              99.609
STRENGTH DESCRIPTION     70.945
ROUTE DESCRIPTION        70.945
NDC_CD                   69.653
DRUG_NM                  69.612
DIAG_003_DIAG_DESC       66.737
DIAG_CD_003              66.031
DIAG_002_DIAG_DESC       56.914
DIAG_CD_002              55.957
DIAG_001_DIAG_DESC       41.624
DIAG_CD_001              40.634
provider_number          34.493
PROC_DESC                22.933
HCPCS                    20.763
DIAG_DESC                15.287
PRI_DIAG_CD              14.068
member_number             0.000
c_allowed                 0.000
C_UTIL_CT                 0.000
Med_Cost_Category         0.000
days_since_earliest_dt    0.000
dtype: float64

In [332]:
# check number of unique values per column
for column in claims.columns:
        print(f"Column: {column}")
        print(claims[column].value_counts())
        print("\n")

Column: PRI_DIAG_CD
PRI_DIAG_CD
Z5111      270903
C50412      45262
C50411      44235
C50912      43217
C50911      41851
            ...  
S22028A         1
S42121D         1
S37812A         1
Q279            1
S12101D         1
Name: count, Length: 9637, dtype: int64


Column: ADM_DIAG_CD
ADM_DIAG_CD
M6281      476
C189       339
S52501D    257
G9340      192
Z48815     179
          ... 
I4821        1
C049         1
B999         1
G720         1
S129XXA      1
Name: count, Length: 1183, dtype: int64


Column: DIAG_CD_001
DIAG_CD_001
Z170       121912
I10         36208
C20         29603
Z171        27399
C50411      26684
            ...  
D3002           1
S62304D         1
M25342          1
T498X5A         1
T560X1S         1
Name: count, Length: 8515, dtype: int64


Column: DIAG_CD_002
DIAG_CD_002
R112       112390
Z170        90746
I10         37726
Z171        36835
C787        21763
            ...  
M21921          1
L651            1
Z83711          1
A609            1
S9215

In [333]:
# check descriptive statistics for numeric columns
pd.set_option('display.float_format', lambda x: '%.3f' % x)
claims.describe()

# DRG: DRG code
# C_UTIL_CT: total number of services
# c_allowed: total allowed amount
# days_since_earliest_dt: days since earliest cancer-related claim
# provider_number: provider number


,DRG,C_UTIL_CT,c_allowed,days_since_earliest_dt,provider_number
count,4891.000,2517387.000,2517387.000,2517387.000,1649058.000
mean,428.134,8.297,269.178,704.547,3130.784
std,251.810,21.382,1883.226,822.841,1923.689
min,1.000,-1.001,-0.020,-366.000,1.000
25%,196.000,0.000,3.321,91.000,2236.000
50%,336.000,1.000,23.200,383.000,2244.000
75%,584.000,1.006,106.049,1127.000,2727.000
max,989.000,365.710,569337.035,3819.000,8220.000


In [334]:
claims.head()

,PRI_DIAG_CD,ADM_DIAG_CD,DIAG_CD_001,DIAG_CD_002,DIAG_CD_003,HCPCS,DRG,Med_Cost_Category,NDC_CD,DRUG_NM,C_UTIL_CT,c_allowed,days_since_earliest_dt,member_number,provider_number,ROUTE DESCRIPTION,STRENGTH DESCRIPTION,PROC_DESC,DIAG_DESC,DRG_DESC,ADM_DIAG_DESC,DIAG_001_DIAG_DESC,DIAG_002_DIAG_DESC,DIAG_003_DIAG_DESC
0,J431,NaN,NaN,NaN,NaN,94727,NaN,Prof PCP Procedures,NaN,NaN,1.000,18.500,-364,A001,NaN,NaN,NaN,Pulm function test by gas,Panlobular emphysema,NaN,NaN,NaN,NaN,NaN
1,J431,NaN,NaN,NaN,NaN,94060,NaN,Prof PCP Procedures,NaN,NaN,1.000,20.000,-364,A001,NaN,NaN,NaN,Evaluation of wheezing,Panlobular emphysema,NaN,NaN,NaN,NaN,NaN
2,J431,NaN,NaN,NaN,NaN,94060,NaN,OP Other,NaN,NaN,1.000,458.120,-364,A001,2236.000,NaN,NaN,Evaluation of wheezing,Panlobular emphysema,NaN,NaN,NaN,NaN,NaN
3,J431,NaN,NaN,NaN,NaN,94726,NaN,OP Other,NaN,NaN,0.000,0.000,-364,A001,2236.000,NaN,NaN,Pulm funct tst plethysmograp,Panlobular emphysema,NaN,NaN,NaN,NaN,NaN
4,J431,NaN,NaN,NaN,NaN,94729,NaN,OP Other,NaN,NaN,0.000,0.000,-364,A001,2236.000,NaN,NaN,Co/membane diffuse capacity,Panlobular emphysema,NaN,NaN,NaN,NaN,NaN


In [335]:
# create a PK for claims
claims = claims.reset_index(drop=True)
claims["claim_id"] = claims.index.astype(str)
claims.head()

,PRI_DIAG_CD,ADM_DIAG_CD,DIAG_CD_001,DIAG_CD_002,DIAG_CD_003,HCPCS,DRG,Med_Cost_Category,NDC_CD,DRUG_NM,C_UTIL_CT,c_allowed,days_since_earliest_dt,member_number,provider_number,ROUTE DESCRIPTION,STRENGTH DESCRIPTION,PROC_DESC,DIAG_DESC,DRG_DESC,ADM_DIAG_DESC,DIAG_001_DIAG_DESC,DIAG_002_DIAG_DESC,DIAG_003_DIAG_DESC,claim_id
0,J431,NaN,NaN,NaN,NaN,94727,NaN,Prof PCP Procedures,NaN,NaN,1.000,18.500,-364,A001,NaN,NaN,NaN,Pulm function test by gas,Panlobular emphysema,NaN,NaN,NaN,NaN,NaN,0
1,J431,NaN,NaN,NaN,NaN,94060,NaN,Prof PCP Procedures,NaN,NaN,1.000,20.000,-364,A001,NaN,NaN,NaN,Evaluation of wheezing,Panlobular emphysema,NaN,NaN,NaN,NaN,NaN,1
2,J431,NaN,NaN,NaN,NaN,94060,NaN,OP Other,NaN,NaN,1.000,458.120,-364,A001,2236.000,NaN,NaN,Evaluation of wheezing,Panlobular emphysema,NaN,NaN,NaN,NaN,NaN,2
3,J431,NaN,NaN,NaN,NaN,94726,NaN,OP Other,NaN,NaN,0.000,0.000,-364,A001,2236.000,NaN,NaN,Pulm funct tst plethysmograp,Panlobular emphysema,NaN,NaN,NaN,NaN,NaN,3
4,J431,NaN,NaN,NaN,NaN,94729,NaN,OP Other,NaN,NaN,0.000,0.000,-364,A001,2236.000,NaN,NaN,Co/membane diffuse capacity,Panlobular emphysema,NaN,NaN,NaN,NaN,NaN,4


In [336]:
# number of unique providers per member number
provider_count = (
    claims.groupby("member_number")["provider_number"]
    .nunique()
    .reset_index(name="num_providers")
)


In [337]:
# provider spend per member
provider_spend = (
    claims.groupby(["member_number", "provider_number"])["c_allowed"]
    .sum()
    .reset_index()
)


In [338]:
# top provider share per member
top_provider_share = (
    provider_spend.sort_values(["member_number", "c_allowed"], ascending=False)
    .groupby("member_number")
    .first()["c_allowed"]
)

In [339]:
# total spend per member
total_spend = claims.groupby("member_number")["c_allowed"].sum()

# provider concentration per member
provider_concentration = (top_provider_share / total_spend).reset_index(name="top_provider_share")

# sort by top provider share
provider_concentration.sort_values("top_provider_share", ascending=False)

,member_number,top_provider_share
2815,C818,1.000
1913,B915,0.992
498,A499,0.985
1259,B261,0.982
748,A749,0.971
...,...,...
1932,B934,NaN
2670,C673,NaN
2750,C753,NaN
3210,D214,NaN


### Initial Join

In [340]:
# patient-level aggregation of cost, utilization, and time since first claim
patient_level = claims.groupby("member_number").agg({
        "c_allowed": "sum",
        "C_UTIL_CT": "sum",
        "days_since_earliest_dt": ["min", "max"]
    })

# rename columns
patient_level.columns = [
    "total_cost",
    "total_util",
    "first_claim_day",
    "last_claim_day"
]

# time since first claim
patient_level["time_since_first_claim"] = patient_level["last_claim_day"] - patient_level["first_claim_day"]

patient_level

,total_cost,total_util,first_claim_day,last_claim_day,time_since_first_claim
member_number,,,,,
A001,34347.750,3635.000,-364,1,365
A002,510592.723,4924.409,-357,1248,1605
A003,292610.070,23653.000,-364,1580,1944
A004,1570.131,891.586,-14,195,209
A005,205847.260,7610.728,-364,1026,1390
...,...,...,...,...,...
D656,631.394,157.597,-41,2,43
D657,58470.372,1190.183,-1,111,112
D658,20523.931,95.685,0,97,97


In [341]:
# category-level spend per member
category_spend = claims.groupby(["member_number", "Med_Cost_Category"])["c_allowed"].sum().unstack(fill_value=0).reset_index()

category_spend

Med_Cost_Category,member_number,IP Behavioral Health,IP Maternity,IP Medical,IP SNF,IP Surgical,OP Behavioral Health,OP ER,OP Observation,OP Other,OP Path/Lab,OP Pharmacy,OP Radiology,OP Surgery,PDN/Home Health,Prof Invisible Prov Procedures,Prof Other,Prof PCP Procedures,Prof Specialist Procedures,RX,Unknown
0,A001,0.000,0.000,6479.460,0.000,0.000,1469.230,0.000,0.000,2577.270,811.780,0.000,228.570,5024.840,0.000,2132.430,383.600,1753.840,5017.300,8469.430,0.000
1,A002,11121.566,0.000,42436.994,0.000,92381.064,0.000,10452.507,8916.282,7333.956,2610.066,120121.367,11498.224,124509.431,1686.759,20627.390,3210.378,7274.885,36204.057,10207.797,0.000
2,A003,0.000,0.000,0.000,0.000,0.000,0.000,684.030,0.000,7247.420,1035.360,72966.420,26884.950,39724.130,979.650,7325.020,1043.210,2641.830,25655.400,106422.650,0.000
3,A004,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,274.128,0.000,0.000,215.830,0.000,0.000,68.180,0.000,0.000,812.753,199.240,0.000
4,A005,0.000,0.000,0.000,16040.090,65092.535,0.000,4948.763,16478.207,162.990,216.341,0.000,0.000,0.000,46910.494,8202.642,16798.127,8305.031,10868.237,11823.801,0.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3652,D656,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,123.241,103.627,0.000,0.000,0.000,0.000,0.000,0.000,253.580,120.612,30.334,0.000
3653,D657,0.000,0.000,0.000,0.000,14477.743,0.000,1046.425,0.000,868.693,311.872,3092.019,24096.903,3270.509,0.000,2857.881,396.204,1300.818,4930.634,1820.672,0.000
3654,D658,0.000,0.000,0.000,0.000,10330.222,0.000,0.000,0.000,156.314,112.559,152.278,0.000,2379.597,0.000,1786.287,0.000,472.021,4297.571,837.083,0.000
3655,D659,0.000,0.000,0.000,0.000,0.000,0.000,1135.433,0.000,1062.573,29.777,0.000,0.000,41631.642,0.000,3446.694,79.361,196.071,9523.914,110.582,0.000


In [342]:
# merge patient_level, category_spend, and staging; use inner join to drop members with no staging records
merged_df = (
    patient_level
    .merge(category_spend, on="member_number", how="left")
    .merge(staging, on="member_number", how="inner")
)
merged_df.drop(columns=["has_path", "has_clin"], inplace=True)

merged_df

,member_number,total_cost,total_util,first_claim_day,last_claim_day,time_since_first_claim,IP Behavioral Health,IP Maternity,IP Medical,IP SNF,IP Surgical,OP Behavioral Health,OP ER,OP Observation,OP Other,OP Path/Lab,OP Pharmacy,OP Radiology,OP Surgery,PDN/Home Health,Prof Invisible Prov Procedures,Prof Other,Prof PCP Procedures,Prof Specialist Procedures,RX,Unknown,ICD10_CODE,MOST_RECENT_PATH_STAGE_DT,MOST_RECENT_CLINICAL_STAGE_DT,PATHOLOGIC_STAGE_GROUP,CLINICAL_STAGE_GROUP,ICD10_group,detailed_cancer_type,cancer_type,FINAL_STAGE,simple_stage
0,A001,34347.750,3635.000,-364,1,365,0.000,0.000,6479.460,0.000,0.000,1469.230,0.000,0.000,2577.270,811.780,0.000,228.570,5024.840,0.000,2132.430,383.600,1753.840,5017.300,8469.430,0.000,C50.111,203.000,25.000,Stage IIA,Stage IIIB,C50,breast,breast,Stage IIIB,Stage 3
1,A002,510592.723,4924.409,-357,1248,1605,11121.566,0.000,42436.994,0.000,92381.064,0.000,10452.507,8916.282,7333.956,2610.066,120121.367,11498.224,124509.431,1686.759,20627.390,3210.378,7274.885,36204.057,10207.797,0.000,C50.412,110.000,NaN,Stage IA,NaN,C50,breast,breast,Stage IA,Stage 1
2,A003,292610.070,23653.000,-364,1580,1944,0.000,0.000,0.000,0.000,0.000,0.000,684.030,0.000,7247.420,1035.360,72966.420,26884.950,39724.130,979.650,7325.020,1043.210,2641.830,25655.400,106422.650,0.000,C50.412,755.000,NaN,Stage IA,NaN,C50,breast,breast,Stage IA,Stage 1
3,A004,1570.131,891.586,-14,195,209,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,274.128,0.000,0.000,215.830,0.000,0.000,68.180,0.000,0.000,812.753,199.240,0.000,C50.812,NaN,-2843.000,NaN,Stage IIA,C50,breast,breast,Stage IIA,Stage 2
4,A005,205847.260,7610.728,-364,1026,1390,0.000,0.000,0.000,16040.090,65092.535,0.000,4948.763,16478.207,162.990,216.341,0.000,0.000,0.000,46910.494,8202.642,16798.127,8305.031,10868.237,11823.801,0.000,C18.7,80.000,NaN,Stage I,NaN,C18,colon,colorectal,Stage I,Stage 1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3636,D656,631.394,157.597,-41,2,43,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,123.241,103.627,0.000,0.000,0.000,0.000,0.000,0.000,253.580,120.612,30.334,0.000,C18.4,-183.000,NaN,Stage IIA,NaN,C18,colon,colorectal,Stage IIA,Stage 2
3637,D657,58470.372,1190.183,-1,111,112,0.000,0.000,0.000,0.000,14477.743,0.000,1046.425,0.000,868.693,311.872,3092.019,24096.903,3270.509,0.000,2857.881,396.204,1300.818,4930.634,1820.672,0.000,C34.12,NaN,4.000,NaN,Stage IIIA,C34,lung,lung,Stage IIIA,Stage 3
3638,D658,20523.931,95.685,0,97,97,0.000,0.000,0.000,0.000,10330.222,0.000,0.000,0.000,156.314,112.559,152.278,0.000,2379.597,0.000,1786.287,0.000,472.021,4297.571,837.083,0.000,C18.7,34.000,NaN,Stage IIA,NaN,C18,colon,colorectal,Stage IIA,Stage 2
3639,D659,57216.047,391.366,0,104,104,0.000,0.000,0.000,0.000,0.000,0.000,1135.433,0.000,1062.573,29.777,0.000,0.000,41631.642,0.000,3446.694,79.361,196.071,9523.914,110.582,0.000,C50.511,39.000,NaN,Stage IA,NaN,C50,breast,breast,Stage IA,Stage 1


In [343]:
# member number showing up multiple times in merged_df, see those members in a table
merged_df["member_number"].value_counts().sort_values(ascending=False)

member_number
B102    3
C794    2
C440    2
C899    2
B091    2
       ..
D622    1
D623    1
D624    1
D625    1
D660    1
Name: count, Length: 3567, dtype: int64

In [344]:
merged_df.columns

Index(['member_number', 'total_cost', 'total_util', 'first_claim_day',
       'last_claim_day', 'time_since_first_claim', 'IP Behavioral Health',
       'IP Maternity', 'IP Medical', 'IP SNF', 'IP Surgical',
       'OP Behavioral Health', 'OP ER', 'OP Observation', 'OP Other',
       'OP Path/Lab', 'OP Pharmacy', 'OP Radiology', 'OP Surgery',
       'PDN/Home Health', 'Prof Invisible Prov Procedures', 'Prof Other',
       'Prof PCP Procedures', 'Prof Specialist Procedures', 'RX', 'Unknown',
       'ICD10_CODE', 'MOST_RECENT_PATH_STAGE_DT',
       'MOST_RECENT_CLINICAL_STAGE_DT', 'PATHOLOGIC_STAGE_GROUP',
       'CLINICAL_STAGE_GROUP', 'ICD10_group', 'detailed_cancer_type',
       'cancer_type', 'FINAL_STAGE', 'simple_stage'],
      dtype='object')

In [345]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3641 entries, 0 to 3640
Data columns (total 36 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   member_number                   3641 non-null   object 
 1   total_cost                      3641 non-null   float64
 2   total_util                      3641 non-null   float64
 3   first_claim_day                 3641 non-null   int64  
 4   last_claim_day                  3641 non-null   int64  
 5   time_since_first_claim          3641 non-null   int64  
 6   IP Behavioral Health            3641 non-null   float64
 7   IP Maternity                    3641 non-null   float64
 8   IP Medical                      3641 non-null   float64
 9   IP SNF                          3641 non-null   float64
 10  IP Surgical                     3641 non-null   float64
 11  OP Behavioral Health            3641 non-null   float64
 12  OP ER                           36

In [346]:
# export merged_df to csv
merged_df.to_csv("merged_staging_claims_patient_level_v2.csv", index=False)